# Panel de análisis ENVI 2020

Este notebook arma dos tablas para el análisis:

- `panel_vivienda`: una fila por vivienda, con todas las columnas de TVIVIENDA y las etiquetas de vivienda del Anexo Técnico.
- `panel_hogar`: una fila por hogar, con los datos de THOGAR, lo que se resume de sus personas (TSDEM) y de su demanda (TDEMANDA), y las etiquetas de la vivienda donde vive.

Llaves: vivienda es `FOLIO + VIV_SEL`, hogar agrega `HOGAR` y persona agrega `N_REN`. El factor de expansión es `FACTOR`.


Pablo Cesar Uriarte

## 1. Configuración

In [66]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:,.2f}".format)
np.set_printoptions(suppress=True)

In [67]:
ROOT = Path("..").resolve()
RAW = ROOT / "2. datos" / "ENVI_2020" / "Bases de datos_crudos"
OUT = ROOT / "2. datos" / "ENVI_2020" / "procesados"
OUT.mkdir(parents=True, exist_ok=True)
DICCIONARIO = RAW / "envi_2020_fd.xlsx"

LLAVE_VIV = ["FOLIO", "VIV_SEL"]
LLAVE_HOG = LLAVE_VIV + ["HOGAR"]
LLAVE_PER = LLAVE_HOG + ["N_REN"]
LLAVE_DEM = LLAVE_HOG + ["C_DENEVI"]
LLAVES = ["FOLIO", "VIV_SEL", "HOGAR", "N_REN", "C_DENEVI"]

UMA_MENSUAL_2020 = 2_641.15
UMBRAL_ING = 5 * UMA_MENSUAL_2020
SM_MENSUAL_2020 = 123.22 * 30.4
UMBRAL_ING_DH = 2 * SM_MENSUAL_2020
PERSONAS_POR_CUARTO_MAX = 2.5
N_MINIMO = 30
ESCENARIO_POSESION = "POSESION_AMPLIA"   # o "POSESION_DOC" para el escenario estricto
K_DONANTES = 5
SEMILLA = 2020

print(f"Umbral de ingreso (5 UMA): ${UMBRAL_ING:,.2f} mensuales")
print(f"Umbral de ingreso individual (2 SM): ${UMBRAL_ING_DH:,.2f} mensuales")

Umbral de ingreso (5 UMA): $13,205.75 mensuales
Umbral de ingreso individual (2 SM): $7,491.78 mensuales


#### Mapas

In [68]:
ENT_NOMBRE = {
    1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur", 4: "Campeche",
    5: "Coahuila de Zaragoza", 6: "Colima", 7: "Chiapas", 8: "Chihuahua", 9: "Ciudad de México",
    10: "Durango", 11: "Guanajuato", 12: "Guerrero", 13: "Hidalgo", 14: "Jalisco", 15: "México",
    16: "Michoacán de Ocampo", 17: "Morelos", 18: "Nayarit", 19: "Nuevo León", 20: "Oaxaca",
    21: "Puebla", 22: "Querétaro", 23: "Quintana Roo", 24: "San Luis Potosí", 25: "Sinaloa",
    26: "Sonora", 27: "Tabasco", 28: "Tamaulipas", 29: "Tlaxcala",
    30: "Veracruz de Ignacio de la Llave", 31: "Yucatán", 32: "Zacatecas",
}
TLOC_NOMBRE = {1: "100,000 y más hab.", 2: "15,000 a 99,999 hab.", 3: "2,500 a 14,999 hab.", 4: "Menos de 2,500 hab."}
TLOC_AMBITO = {1: "Urbano", 2: "Urbano", 3: "Urbano", 4: "Rural"}

# P3_4A, periodicidad del ingreso (semana, quincena, mes, año)
FACTOR_MENSUAL = {1: 52 / 12, 2: 2.0, 3: 1.0, 4: 1 / 12}

TENENCIA = {1: "rentada", 2: "prestada_familiar", 3: "prestada_trabajo", 4: "propia_pagando",
            5: "propia", 6: "intestada_litigio", 7: "otra"}

## 2. Lectura y carga de las tablas crudas

Las tablas se leen como texto para no perder los ceros de las llaves (`0000001`, `01`). También se carga el diccionario de datos de INEGI, que se usa para limpiar los códigos de "no sabe" y para describir las columnas al final.

In [69]:
def leer_tabla(nombre):
    df = pd.read_csv(RAW / f"{nombre}.csv", dtype=str, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
    df.columns = [c.replace("﻿", "").replace('"', "").strip() for c in df.columns]
    return df


LLAVE_TABLA = {"TVIVIENDA": LLAVE_VIV, "THOGAR": LLAVE_HOG, "TSDEM": LLAVE_PER, "TDEMANDA": LLAVE_DEM}
crudas = {t: leer_tabla(t) for t in LLAVE_TABLA}

pd.DataFrame({
    "tabla": list(crudas),
    "llave": [" + ".join(LLAVE_TABLA[t]) for t in crudas],
    "filas": [len(df) for df in crudas.values()],
    "columnas": [df.shape[1] for df in crudas.values()],
})

,tabla,llave,filas,columnas
0,TVIVIENDA,FOLIO + VIV_SEL,55147,416
1,THOGAR,FOLIO + VIV_SEL + HOGAR,56578,14
2,TSDEM,FOLIO + VIV_SEL + HOGAR + N_REN,202595,23
3,TDEMANDA,FOLIO + VIV_SEL + HOGAR + C_DENEVI,13121,22


In [70]:
def leer_diccionario(hoja):
    fd = pd.read_excel(DICCIONARIO, sheet_name=hoja, header=None, dtype=str).iloc[:, 1:7]
    fd.columns = ["pregunta", "columna", "tipo", "tamano", "codigo", "etiqueta"]
    fd["columna"] = fd["columna"].ffill()

    tiene_codigo = fd["codigo"].notna()
    es_referencia = fd["codigo"].str.startswith("[", na=False)
    es_encabezado = fd["columna"].isin(["Nemónico", "[2]"])
    fd = fd[tiene_codigo & ~es_referencia & ~es_encabezado].copy()

    fd["pregunta"] = fd.groupby("columna")["pregunta"].transform("first")
    for c in ["pregunta", "columna", "codigo", "etiqueta"]:
        fd[c] = fd[c].str.strip()
    fd["tabla"] = hoja
    fd["es_rango"] = fd["codigo"].str.match(r"^\d+\s*-\s*\d+$", na=False)
    return fd[["tabla", "columna", "pregunta", "codigo", "etiqueta", "es_rango"]]


dicc = pd.concat([leer_diccionario(t) for t in LLAVE_TABLA], ignore_index=True)
dicc_vars = (dicc.groupby(["tabla", "columna"], sort=False)
                 .agg(pregunta=("pregunta", "first"), es_numerica=("es_rango", "any"))
                 .reset_index())
print(f"{len(dicc_vars):,} variables en el diccionario")

475 variables en el diccionario


## 3. Preparación y limpieza inicial

Las llaves se quedan como texto y lo demás pasa a número entero (`Int64`, que acepta celdas vacías).

En las variables numéricas (montos, edad, años, metros) los códigos de no sabe, no responde y no especificado se cambian a vacío, porque no son cantidades. Los códigos salen del diccionario. En las variables de opciones el 9 de "no sabe" se deja como está.

In [71]:
tablas = {}
for t, df in crudas.items():
    df = df.copy()
    for c in df.columns:
        if c not in LLAVES:
            df[c] = pd.to_numeric(df[c]).astype("Int64")
    tablas[t] = df

numericas = dicc_vars.loc[dicc_vars["es_numerica"], ["tabla", "columna"]]
no_validos = dicc.merge(numericas, on=["tabla", "columna"])
no_validos = no_validos[no_validos["etiqueta"].str.contains("No sabe|No responde|No especificado", case=False, na=False)
                        & no_validos["codigo"].str.isdigit()]

cambios = []
for (t, col), g in no_validos.groupby(["tabla", "columna"]):
    codigos = g["codigo"].astype(int).tolist()
    marcados = tablas[t][col].isin(codigos)
    tablas[t].loc[marcados, col] = pd.NA
    cambios.append([t, col, codigos, int(marcados.sum())])

cambios = pd.DataFrame(cambios, columns=["tabla", "columna", "codigos", "registros"])
cambios.sort_values("registros", ascending=False).head(10)

,tabla,columna,codigos,registros
150,TVIVIENDA,P5_9,"[999999888, 999999999]",9804
3,TSDEM,P3_4,"[99888, 99999]",9504
5,TVIVIENDA,P4_20_1,"[998, 999]",8070
6,TVIVIENDA,P4_21_1,"[998, 999]",7762
4,TVIVIENDA,P4_19_1,"[98, 99]",6159
149,TVIVIENDA,P5_8,"[99888, 99999]",6151
18,TVIVIENDA,P5_16_07,"[999999888, 999999999]",5717
53,TVIVIENDA,P5_24,"[999999888, 999999999]",2028
11,TVIVIENDA,P5_14,"[999999888, 999999999]",2020
28,TVIVIENDA,P5_19_1,"[9999888, 9999999]",1567


In [72]:
tviv, thogar, tsdem, tdemanda = tablas["TVIVIENDA"], tablas["THOGAR"], tablas["TSDEM"], tablas["TDEMANDA"]


def bandera(condicion):
    return condicion.fillna(False).astype(int)

## 4. Validación de las llaves

Antes de cruzar tablas se revisa que cada llave sea única y que las tablas hablen de las mismas viviendas y hogares. Si algo falla, el notebook se detiene.

In [73]:
def llaves(df, cols):
    return set(df[cols].itertuples(index=False, name=None))


verificaciones = {}

# Cada tabla tiene una sola fila por llave
for t, llave in LLAVE_TABLA.items():
    verificaciones[f"{t} sin llaves repetidas"] = not tablas[t].duplicated(llave).any()

# THOGAR y TVIVIENDA tienen las mismas viviendas
verificaciones["Mismas viviendas en THOGAR y TVIVIENDA"] = llaves(thogar, LLAVE_VIV) == llaves(tviv, LLAVE_VIV)

# THOGAR y TSDEM tienen los mismos hogares
verificaciones["Mismos hogares en THOGAR y TSDEM"] = llaves(thogar, LLAVE_HOG) == llaves(tsdem, LLAVE_HOG)

# TDEMANDA solo trae hogares que dijeron tener necesidad de vivienda
con_necesidad = llaves(thogar[thogar["P3A1_1"] == 1], LLAVE_HOG)
verificaciones["TDEMANDA solo con hogares P3A1_1 = 1"] = llaves(tdemanda, LLAVE_HOG) <= con_necesidad

# Cada hogar tiene un solo jefe o jefa
jefes = tsdem[tsdem["PAREN"] == 1].groupby(LLAVE_HOG).size()
verificaciones["Un jefe por hogar"] = bool(len(jefes) == len(thogar) and (jefes == 1).all())

# El FACTOR es el mismo para toda la vivienda en las cuatro tablas
factores = pd.concat([tablas[t][LLAVE_VIV + ["FACTOR"]] for t in LLAVE_TABLA]).drop_duplicates()
verificaciones["FACTOR único por vivienda"] = not factores.duplicated(LLAVE_VIV).any()

verificaciones = pd.Series(verificaciones).map({True: "OK", False: "FALLA"}).to_frame("resultado")
assert (verificaciones["resultado"] == "OK").all(), "Hay llaves con problemas"

print(f"viviendas: {len(tviv):,} | hogares: {len(thogar):,} | personas: {len(tsdem):,} | demandas: {len(tdemanda):,}")
verificaciones

viviendas: 55,147 | hogares: 56,578 | personas: 202,595 | demandas: 13,121


,resultado
TVIVIENDA sin llaves repetidas,OK
THOGAR sin llaves repetidas,OK
TSDEM sin llaves repetidas,OK
TDEMANDA sin llaves repetidas,OK
Mismas viviendas en THOGAR y TVIVIENDA,OK
Mismos hogares en THOGAR y TSDEM,OK
TDEMANDA solo con hogares P3A1_1 = 1,OK
Un jefe por hogar,OK
FACTOR único por vivienda,OK


## 5. Cuadre contra las cifras oficiales

Los totales expandidos con `FACTOR` se comparan con los tabulados de INEGI. La diferencia debe ser cero. Hogares se cuentan en THOGAR y viviendas en TVIVIENDA.

In [74]:
tot_hog = thogar["FACTOR"].sum()
hog_nec = thogar.loc[thogar["P3A1_1"] == 1, "FACTOR"].sum()
tot_viv = tviv["FACTOR"].sum()
viv_rentadas = tviv.loc[tviv["P5_1"] == 1, "FACTOR"].sum()

cuadre = pd.DataFrame({
    "indicador": ["Total de hogares", "Hogares con necesidad de vivienda (P3A1_1=1)",
                  "Total de viviendas", "Viviendas rentadas (P5_1=1)"],
    "fuente": ["Cuadro 5.1", "Cuadro 5.1", "Cuadro 1.1", "Cuadro 3.1"],
    "oficial": [36_210_467, 7_628_562, 35_259_433, 5_782_349],
    "calculado": [tot_hog, hog_nec, tot_viv, viv_rentadas],
})
cuadre["diferencia"] = cuadre["calculado"] - cuadre["oficial"]
cuadre

,indicador,fuente,oficial,calculado,diferencia
0,Total de hogares,Cuadro 5.1,36210467,36210467,0
1,Hogares con necesidad de vivienda (P3A1_1=1),Cuadro 5.1,7628562,7628562,0
2,Total de viviendas,Cuadro 1.1,35259433,35259433,0
3,Viviendas rentadas (P5_1=1),Cuadro 3.1,5782349,5782349,0


## 6. Variables de personas (TSDEM) al hogar

Ingreso. `P3_4` es lo que gana cada persona por su trabajo y `P3_4A` cada cuánto lo recibe. Se pasa a mensual y se suma en el hogar (`ING_HOG`). Los códigos 99999 (no sabe) y 99888 (no responde) ya quedaron vacíos en la sección 3. Aquí se marcan con `ing_no_reportado` en la persona y con `ingreso_incompleto` en el hogar.

Derechohabiencia. `P3_5` es el derecho a crédito de vivienda por el trabajo: 1 INFONAVIT, 2 FOVISSSTE, 3 otra institución, 4 ninguno. A su lado se agrega `P3_5_derechohab`, que vale 1 si la persona tiene cualquiera de los tres. En el hogar, `DERECHO_CRED` marca si alguien tiene INFONAVIT o FOVISSSTE, como pide el Anexo.

Ingreso individual. `ING_IND` es el ingreso mensual de la persona con derecho a crédito INFONAVIT o FOVISSSTE; si hay más de una, se toma el mayor. `ING_IND_DH` marca a los hogares derechohabientes donde ese ingreso es de 2 salarios mínimos o menos ($7,491.78). `ING_CONFIABLE` marca a los hogares con al menos un perceptor y sin ingresos faltantes.

Jefatura y grupos prioritarios. Se pasan al hogar los datos del jefe o jefa y se marcan los grupos del programa: jefatura femenina, jefe joven (18 a 29 años), algún integrante de 60 años o más, algún integrante indígena y algún integrante con limitación permanente.

In [75]:
p = tsdem.copy()

p["ing_mensual"] = (p["P3_4"] * p["P3_4A"].map(FACTOR_MENSUAL)).astype("Float64")
p.loc[p["P3_4"].isin([0]), "ing_mensual"] = 0
p["ing_no_reportado"] = bandera(crudas["TSDEM"]["P3_4"].isin(["99888", "99999"])
                                | ((p["P3_4"] > 0) & p["P3_4A"].isin([9])))
p["perceptor"] = bandera(p["ing_mensual"] > 0)

p.insert(p.columns.get_loc("P3_5") + 1, "P3_5_derechohab", bandera(p["P3_5"].isin([1, 2, 3])))
p["infonavit_fovissste"] = bandera(p["P3_5"].isin([1, 2]))
p["ing_dh"] = p["ing_mensual"].where(p["infonavit_fovissste"] == 1)

p["indigena"] = bandera((p["P2_5"] == 1) | (p["P2_6"] == 1))
p["limitacion"] = bandera(p["P3_1"] == 7)
p["mayor_60"] = bandera(p["EDAD"] >= 60)

p[["EDAD", "P3_4", "P3_4A", "ing_mensual", "ing_no_reportado", "P3_5", "P3_5_derechohab"]].head()

,EDAD,P3_4,P3_4A,ing_mensual,ing_no_reportado,P3_5,P3_5_derechohab
0,61,<NA>,<NA>,<NA>,1,4,0
1,64,<NA>,<NA>,<NA>,0,<NA>,0
2,67,3500,3,"3,500.00",0,4,0
3,74,3000,1,"13,000.00",0,4,0
4,72,<NA>,<NA>,<NA>,0,<NA>,0


In [76]:
hog_personas = p.groupby(LLAVE_HOG).agg(
    tam_hogar=("N_REN", "size"),
    n_mayores60=("mayor_60", "sum"),
    n_perceptores=("perceptor", "sum"),
    n_ing_no_reportado=("ing_no_reportado", "sum"),
    ING_HOG=("ing_mensual", "sum"),
    n_derechohab=("P3_5_derechohab", "sum"),
    DERECHO_CRED=("infonavit_fovissste", "max"),
    ING_IND=("ing_dh", "max"),
    hogar_indigena=("indigena", "max"),
    hogar_limitacion=("limitacion", "max"),
).reset_index()

hog_personas.insert(hog_personas.columns.get_loc("n_derechohab") + 1, "hogar_derechohab",
                    bandera(hog_personas["n_derechohab"] > 0))
hog_personas["ingreso_incompleto"] = bandera(hog_personas["n_ing_no_reportado"] > 0)
hog_personas["ING_HOG_UMA"] = hog_personas["ING_HOG"] / UMA_MENSUAL_2020
hog_personas["ING_5UMA"] = bandera(hog_personas["ING_HOG"] <= UMBRAL_ING)
hog_personas["ING_IND_DH"] = bandera((hog_personas["DERECHO_CRED"] == 1) & (hog_personas["ING_IND"] <= UMBRAL_ING_DH))
hog_personas["ING_CONFIABLE"] = bandera((hog_personas["n_perceptores"] > 0) & (hog_personas["ingreso_incompleto"] == 0))

In [77]:
COLS_JEFE = ["SEXO", "EDAD", "NIV", "P2_8", "P3_1", "P3_3", "P3_5"]
jefe = (p.loc[p["PAREN"] == 1, LLAVE_HOG + COLS_JEFE]
          .rename(columns={c: f"JEFE_{c}" for c in COLS_JEFE}))

# Cruce: cada hogar con su jefe o jefa, uno a uno
hog_personas = hog_personas.merge(jefe, on=LLAVE_HOG, how="left", validate="one_to_one")

hog_personas["MAYOR18"] = bandera(hog_personas["JEFE_EDAD"] >= 18)
hog_personas["gp_mujer"] = bandera(hog_personas["JEFE_SEXO"] == 2)
hog_personas["gp_joven"] = bandera(hog_personas["JEFE_EDAD"].between(18, 29))
hog_personas["gp_adulto_mayor"] = bandera(hog_personas["n_mayores60"] > 0)
hog_personas["gp_indigena"] = hog_personas["hogar_indigena"]
hog_personas["gp_discapacidad"] = hog_personas["hogar_limitacion"]

print(hog_personas.shape)
hog_personas.head()

(56578, 32)


,FOLIO,VIV_SEL,HOGAR,tam_hogar,n_mayores60,n_perceptores,n_ing_no_reportado,ING_HOG,n_derechohab,hogar_derechohab,DERECHO_CRED,ING_IND,hogar_indigena,hogar_limitacion,ingreso_incompleto,ING_HOG_UMA,ING_5UMA,ING_IND_DH,ING_CONFIABLE,JEFE_SEXO,JEFE_EDAD,JEFE_NIV,JEFE_P2_8,JEFE_P3_1,JEFE_P3_3,JEFE_P3_5,MAYOR18,gp_mujer,gp_joven,gp_adulto_mayor,gp_indigena,gp_discapacidad
0,0000001,01,1,3,3,1,1,"3,500.00",0,0,0,<NA>,1,0,1,1.33,1,0,0,1,61,3,6,2,4,4,1,0,0,1,1,0
1,0000001,02,1,3,2,2,0,"19,500.00",1,1,1,"6,500.00",0,0,0,7.38,0,1,1,1,74,2,5,1,4,4,1,0,0,1,0,0
2,0000001,03,1,1,0,0,1,0.00,0,0,0,<NA>,1,0,1,0.00,1,0,0,2,57,8,3,1,4,4,1,1,0,0,1,0
3,0000001,04,1,8,1,0,0,0.00,0,0,0,<NA>,0,0,0,0.00,1,0,0,2,55,4,5,6,<NA>,<NA>,1,1,0,1,0,0
4,0000001,05,1,3,1,2,0,"23,666.67",1,1,1,"15,000.00",0,0,0,8.96,0,0,1,2,86,8,4,4,<NA>,<NA>,1,1,0,1,0,0


In [78]:
# Revisión del ingreso no reportado, por persona y por hogar
con_pregunta = crudas["TSDEM"]["P3_4"].notna()
incompletos = hog_personas["ingreso_incompleto"] == 1
dudosos = incompletos & (hog_personas["ING_5UMA"] == 1)

print(f"Personas con pregunta de ingreso:         {con_pregunta.sum():,}")
print(f"Personas con ingreso no reportado:        {p['ing_no_reportado'].sum():,}")
print(f"Hogares con ingreso incompleto:           {incompletos.sum():,} de {len(hog_personas):,}")
print(f"Hogares incompletos que hoy dan 5 UMA o menos: {dudosos.sum():,}")

Personas con pregunta de ingreso:         82,671
Personas con ingreso no reportado:        9,522
Hogares con ingreso incompleto:           6,682 de 56,578
Hogares incompletos que hoy dan 5 UMA o menos: 6,099


#### Ingreso no reportado

9,522 personas (11.5% de quienes tienen la pregunta) no dieron un monto de ingreso usable, lo que deja 6,682 hogares con ingreso incompleto, 4,006 de ellos en cero porque nadie reportó. Esos hogares parecen más pobres de lo que son y tienden a entrar en el umbral de 5 UMA. 

Como alternativa se imputa con PMM. Primero se estima una regresión del logaritmo del ingreso con quienes sí lo reportaron, tomando en cuenta sexo, edad, escolaridad, posición en el trabajo, derechohabiencia, entidad y tamaño de localidad. Se usa el logaritmo porque muchos ganan poco y pocos ganan mucho, y así esa diferencia se suaviza. Con la regresión se calcula un ingreso esperado para cada persona, y a quien no tiene dato se le copia el ingreso real de una de las 5 personas con el ingreso esperado más parecido. El resultado queda en `ING_HOG_PMM` e `ING_5UMA_PMM`, sin modificar el ingreso observado.

In [79]:
p["ing_mensual_pmm"] = p["ing_mensual"]
p.loc[(p["ing_no_reportado"] == 1) & p["P3_3"].isin([1]), "ing_mensual_pmm"] = 0

donantes = bandera((p["ing_mensual"] > 0) & (p["P3_4"] < 98_000)) == 1
receptores = (p["ing_no_reportado"] == 1) & ~p["P3_3"].isin([1])
modelo = p[donantes | receptores]

X = pd.DataFrame({
    "sexo": modelo["SEXO"],
    "edad": pd.cut(modelo["EDAD"], [0, 24, 34, 44, 54, 64, 120]),
    "escolaridad": modelo["NIV"],
    "posicion": modelo["P3_3"],
    "derechohab": modelo["P3_5"],
    "ent": modelo["ENT"],
    "tloc": modelo["TLOC"],
}).astype(str)
X = pd.get_dummies(X, drop_first=True).astype(float)
X.insert(0, "constante", 1.0)

es_donante = donantes[modelo.index].values
es_receptor = receptores[modelo.index].values
log_ingreso = np.log(modelo.loc[es_donante, "ing_mensual"].astype(float).values)
beta = np.linalg.lstsq(X.values[es_donante], log_ingreso, rcond=None)[0]
prediccion = X.values @ beta

orden = np.argsort(prediccion[es_donante])
pred_donantes = prediccion[es_donante][orden]
ing_donantes = modelo.loc[es_donante, "ing_mensual"].astype(float).values[orden]

rng = np.random.default_rng(SEMILLA)
posicion = np.searchsorted(pred_donantes, prediccion[es_receptor])
inicio = np.clip(posicion - K_DONANTES // 2, 0, len(pred_donantes) - K_DONANTES)
elegido = inicio + rng.integers(0, K_DONANTES, size=len(inicio))
p.loc[receptores, "ing_mensual_pmm"] = ing_donantes[elegido]

# Cruce: ingreso con PMM de cada hogar, uno a uno
pmm_hogar = p.groupby(LLAVE_HOG)["ing_mensual_pmm"].sum().rename("ING_HOG_PMM").reset_index()
hog_personas = hog_personas.merge(pmm_hogar, on=LLAVE_HOG, how="left", validate="one_to_one")
hog_personas["ING_5UMA_PMM"] = bandera(hog_personas["ING_HOG_PMM"] <= UMBRAL_ING)

# Comprobación: todos los receptores tienen ingreso y los imputados se parecen a los reportados
assert p.loc[receptores, "ing_mensual_pmm"].notna().all()
print(f"Donantes: {es_donante.sum():,} | receptores: {es_receptor.sum():,}")
print(f"Mediana reportada: ${np.median(ing_donantes):,.0f} | mediana imputada: ${np.median(ing_donantes[elegido]):,.0f}")
factor = p.groupby(LLAVE_HOG)["FACTOR"].first().values
completos = hog_personas["ingreso_incompleto"] == 0
pd.Series({
    "ingreso observado, todos los hogares": np.average(hog_personas["ING_5UMA"], weights=factor) * 100,
    "ingreso observado, solo hogares completos": np.average(hog_personas.loc[completos, "ING_5UMA"], weights=factor[completos]) * 100,
    "ingreso con PMM, todos los hogares": np.average(hog_personas["ING_5UMA_PMM"], weights=factor) * 100,
}).round(1).to_frame("% hogares en 5 UMA o menos")

Donantes: 72,063 | receptores: 9,522
Mediana reportada: $5,633 | mediana imputada: $6,500


,% hogares en 5 UMA o menos
"ingreso observado, todos los hogares",77.50
"ingreso observado, solo hogares completos",75.50
"ingreso con PMM, todos los hogares",72.40


## 7. Variables de hogar, demanda y vivienda según el Anexo Técnico

Hogar (THOGAR)

- `NEC_DECLARADA`: el hogar dice que necesita o planea rentar, comprar o construir (P3A1_1 = 1).
- `HOG_SEC`: hogar secundario, no es el hogar del dueño ni de quien paga la renta (P4_1 = 2).
- `PROP_OTRA`: alguien del hogar es dueño de otra vivienda (P7_1 = 1).

Demanda (TDEMANDA), se marca si alguna de las viviendas que busca el hogar cumple

- `DEM_RENTA`: quiere rentar (P3A4 = 1).
- `DEM_COMPRA`: quiere comprar nueva o usada (P3A4 = 2 o 3).
- `DEM_CONSTRUIR`: quiere construir (P3A4 = 4).
- `DEM_CONSTR_TERR`: quiere construir y ya tiene terreno (P3A4 = 4 y P3A7 = 1).
- `PLAN_CRED`: piensa usar crédito de INFONAVIT, FOVISSSTE, banco u otra institución pública (P3A8 = 2, 3, 4 o 5).
- `PLAN_AHORRO`: piensa usar recursos propios (P3A8 = 1). Aproxima el ahorro previo.

Vivienda (TVIVIENDA)

- `MAT_DET`: paredes de desecho, cartón, carrizo o embarro (P4_4 = 1, 2, 4, 5) o techo de desecho, cartón o palma (P4_5 = 1, 2, 6).
- `MAT_REG`: paredes de lámina o madera (P4_4 = 3, 6), techo de lámina, madera o teja (P4_5 = 3, 4, 7, 9) o piso de tierra (P4_6 = 1). El adobe no cuenta.
- `SIN_EXC`: no tiene excusado (P4_11 = 2).
- `HACIN`: más de 2.5 personas por cuarto (P1_1 entre P4_10A).
- `REZAGO`: tiene alguna de las cuatro anteriores.
- `CAR_SERV`: sanitario sin agua, sin agua entubada, drenaje a barranca o sin drenaje, sin luz, o cocina con leña sin chimenea.
- `SIN_COCINA`: no tiene cuarto para cocinar (P4_9 = 2).
- `DANO_EST`: grietas, deformaciones, hundimientos, humedad o daño en columnas (P4_25_1 a P4_25_5).
- `NEC_MEJ_DECL` y `NEC_AMP_DECL`: el hogar dice que necesita reparar o ampliar (P6_1).
- `NEC_ACCES`: necesita adaptaciones para personas con discapacidad (P6_7).
- `VIV_PROPIA`: propia o se está pagando (P5_1 = 4 o 5). `POSESION_DOC`: además tiene escritura a nombre del dueño (P5_7 = 1).
- `TEN_IRREG`: intestada o en litigio (P5_1 = 6), o propia sin escritura o a nombre de otra persona (P5_7 = 2 o 3).
- `MULTIFAM`: departamento, vecindad o cuarto de azotea (P4_3 = 4, 5, 6). `DANO_COMUN`: daños en pasillos, escaleras o elevador (P4_26).
- `UH_SOCIAL`: vivienda multifamiliar comprada con crédito de INFONAVIT o FOVISSSTE (P5_15_01 o P5_15_02 = 1). Aproxima la unidad habitacional de interés social.
- `CRED_VIGENTE`: tiene deuda vigente con el crédito INFONAVIT de la vivienda (P5_18_1 = 1).
- `APOYO_PREVIO`: en 2019 o 2020 el gobierno le dio material gratis para construir, ampliar, rehabilitar o mantener la vivienda (P5_33 y P5_34 para 2019, P5_46 y P5_47 para 2020). No capta subsidios en dinero.

In [80]:
h = thogar.copy()
h["ENT_NOMBRE"] = h["ENT"].map(ENT_NOMBRE)
h["TLOC_NOMBRE"] = h["TLOC"].map(TLOC_NOMBRE)
h["AMBITO"] = h["TLOC"].map(TLOC_AMBITO)
h["NEC_DECLARADA"] = bandera(h["P3A1_1"] == 1)
h["HOG_SEC"] = bandera(h["P4_1"] == 2)
h["PROP_OTRA"] = bandera(h["P7_1"] == 1)

In [81]:
d = tdemanda.copy()
fuentes = d[["P3A8_1", "P3A8_2", "P3A8_3"]]

d["DEM_RENTA"] = bandera(d["P3A4"] == 1)
d["DEM_COMPRA"] = bandera(d["P3A4"].isin([2, 3]))
d["DEM_CONSTRUIR"] = bandera(d["P3A4"] == 4)
d["DEM_CONSTR_TERR"] = bandera((d["P3A4"] == 4) & (d["P3A7"] == 1))
d["PLAN_CRED"] = bandera(fuentes.isin([2, 3, 4, 5]).any(axis=1))
d["PLAN_AHORRO"] = bandera(fuentes.isin([1]).any(axis=1))

BANDERAS_DEM = ["DEM_RENTA", "DEM_COMPRA", "DEM_CONSTRUIR", "DEM_CONSTR_TERR", "PLAN_CRED", "PLAN_AHORRO"]
hog_dem = d.groupby(LLAVE_HOG)[BANDERAS_DEM].max().reset_index()
hog_dem.insert(3, "n_demandas", d.groupby(LLAVE_HOG).size().values)
hog_dem.head()

,FOLIO,VIV_SEL,HOGAR,n_demandas,DEM_RENTA,DEM_COMPRA,DEM_CONSTRUIR,DEM_CONSTR_TERR,PLAN_CRED,PLAN_AHORRO
0,0000001,05,1,1,0,0,1,0,0,0
1,0000003,01,1,1,0,1,0,0,1,0
2,0000004,03,1,1,0,1,0,0,1,0
3,0000005,01,1,1,0,1,0,0,1,0
4,0000005,05,1,1,0,1,0,0,1,0


In [82]:
v = tviv.copy()


def alguno(cols):
    return v[cols].isin([1]).any(axis=1)


v["personas_por_cuarto"] = v["P1_1"] / v["P4_10A"]
v["MAT_DET"] = bandera(v["P4_4"].isin([1, 2, 4, 5]) | v["P4_5"].isin([1, 2, 6]))
v["MAT_REG"] = bandera(v["P4_4"].isin([3, 6]) | v["P4_5"].isin([3, 4, 7, 9]) | v["P4_6"].isin([1]))
v["SIN_EXC"] = bandera(v["P4_11"] == 2)
v["HACIN"] = bandera(v["personas_por_cuarto"] > PERSONAS_POR_CUARTO_MAX)
v["REZAGO"] = v[["MAT_DET", "MAT_REG", "SIN_EXC", "HACIN"]].max(axis=1)
v["CAR_SERV"] = bandera((v["P4_12"] == 3) | (v["P4_13"] == 3) | v["P4_15"].isin([3, 4, 5])
                        | (v["P4_16"] == 2) | ((v["P4_17"] == 2) & (v["P4_18"] == 2)))
v["SIN_COCINA"] = bandera(v["P4_9"] == 2)
v["DANO_EST"] = bandera(alguno(["P4_25_1", "P4_25_2", "P4_25_3", "P4_25_4", "P4_25_5"]))
v["NEC_MEJ_DECL"] = bandera(alguno(["P6_1_1", "P6_1_5", "P6_1_6", "P6_1_7", "P6_1_9"]))
v["NEC_AMP_DECL"] = bandera(alguno(["P6_1_2", "P6_1_3", "P6_1_4", "P6_1_8"]))
v["NEC_ACCES"] = bandera(alguno(["P6_7_1", "P6_7_2", "P6_7_3", "P6_7_4"]))

v["tenencia"] = v["P5_1"].map(TENENCIA)
v["VIV_PROPIA"] = bandera(v["P5_1"].isin([4, 5]))
v["POSESION_DOC"] = bandera((v["VIV_PROPIA"] == 1) & (v["P5_7"] == 1))
v["TEN_IRREG"] = bandera((v["P5_1"] == 6) | ((v["VIV_PROPIA"] == 1) & v["P5_7"].isin([2, 3])))

v["MULTIFAM"] = bandera(v["P4_3"].isin([4, 5, 6]))
v["DANO_COMUN"] = bandera(alguno(["P4_26_1", "P4_26_2", "P4_26_3"]))
v["UH_SOCIAL"] = bandera((v["MULTIFAM"] == 1) & ((v["P5_15_01"] == 1) | (v["P5_15_02"] == 1)))

v["CRED_VIGENTE"] = bandera(v["P5_18_1"] == 1)
apoyo_2019 = [(v[f"P5_33_{k}"] == 1) & (v[f"P5_34_{k}_1"] == 1) for k in range(1, 6)]
apoyo_2020 = [(v[f"P5_46_{k}"] == 1) & (v[f"P5_47_{k}_1"] == 1) for k in range(1, 6)]
v["APOYO_PREVIO"] = bandera(pd.concat(apoyo_2019 + apoyo_2020, axis=1).any(axis=1))

# Cruce: número de hogares en cada vivienda, contado en THOGAR
n_hogares = thogar.groupby(LLAVE_VIV).size().rename("n_hogares").reset_index()
v = v.merge(n_hogares, on=LLAVE_VIV, how="left", validate="one_to_one")

ETIQUETAS_VIV = ["n_hogares", "personas_por_cuarto", "MAT_DET", "MAT_REG", "SIN_EXC", "HACIN", "REZAGO",
                 "CAR_SERV", "SIN_COCINA", "DANO_EST", "NEC_MEJ_DECL", "NEC_AMP_DECL", "NEC_ACCES",
                 "tenencia", "VIV_PROPIA", "POSESION_DOC", "TEN_IRREG", "MULTIFAM", "DANO_COMUN",
                 "UH_SOCIAL", "CRED_VIGENTE", "APOYO_PREVIO"]

# Comprobación: porcentaje de viviendas (expandido) con cada etiqueta
binarias = [c for c in ETIQUETAS_VIV if v[c].isin([0, 1]).all()]
(v[binarias].mul(v["FACTOR"], axis=0).sum() / v["FACTOR"].sum() * 100).round(1).to_frame("% viviendas")

,% viviendas
MAT_DET,1.40
MAT_REG,21.00
SIN_EXC,2.40
HACIN,4.20
REZAGO,24.10
CAR_SERV,14.10
SIN_COCINA,10.80
DANO_EST,59.80
NEC_MEJ_DECL,40.60
NEC_AMP_DECL,39.30


## 8. Integración de los paneles

`panel_vivienda` es TVIVIENDA completa más sus etiquetas.

`panel_hogar` parte de THOGAR y se le pega lo de las personas, la demanda y las etiquetas de la vivienda (no las 400 columnas crudas de TVIVIENDA, que se consultan en `panel_vivienda` con `FOLIO + VIV_SEL`). Después se crean las etiquetas que mezclan hogar y vivienda:

- `SIN_VIV_PROPIA`: la vivienda es rentada, prestada u otra situación (P5_1 = 1, 2, 3, 7), o el hogar es secundario.
- `DEM_VIV`: el hogar declaró necesidad y quiere rentar, comprar o construir.
- `NEC_VIV`: tiene `SIN_VIV_PROPIA` o `DEM_VIV`.
- `NEC_VIV_OFICIAL`: hogar secundario en vivienda sin rezago (referencia de CONAVI).
- `POSESION_AMPLIA`: igual a `VIV_PROPIA`, con o sin escritura.
- `gp_discapacidad`: se completa con `NEC_ACCES` de la vivienda.
- `grupo_prioritario`: pertenece a alguno de los cinco grupos.

In [83]:
panel_vivienda = v

# Cruce 1: hogar con el resumen de sus personas, uno a uno
panel_hogar = h.merge(hog_personas, on=LLAVE_HOG, how="left", validate="one_to_one")

# Cruce 2: hogar con su demanda; los hogares sin demanda quedan en 0
panel_hogar = panel_hogar.merge(hog_dem, on=LLAVE_HOG, how="left", validate="one_to_one")
panel_hogar[["n_demandas"] + BANDERAS_DEM] = panel_hogar[["n_demandas"] + BANDERAS_DEM].fillna(0).astype(int)

# Cruce 3: cada hogar recibe las etiquetas de su vivienda, muchos a uno
panel_hogar = panel_hogar.merge(v[LLAVE_VIV + ["P5_1"] + ETIQUETAS_VIV], on=LLAVE_VIV, how="left",
                                validate="many_to_one")

# Comprobación: los cruces no agregaron ni quitaron hogares
assert len(panel_hogar) == len(thogar)
assert panel_hogar["FACTOR"].sum() == tot_hog

In [84]:
panel_hogar["SIN_VIV_PROPIA"] = bandera(panel_hogar["P5_1"].isin([1, 2, 3, 7]) | (panel_hogar["HOG_SEC"] == 1))
panel_hogar["DEM_VIV"] = bandera((panel_hogar["NEC_DECLARADA"] == 1)
                                 & (panel_hogar[["DEM_RENTA", "DEM_COMPRA", "DEM_CONSTRUIR"]].max(axis=1) == 1))
panel_hogar["NEC_VIV"] = bandera((panel_hogar["SIN_VIV_PROPIA"] == 1) | (panel_hogar["DEM_VIV"] == 1))
panel_hogar["NEC_VIV_OFICIAL"] = bandera((panel_hogar["HOG_SEC"] == 1) & (panel_hogar["REZAGO"] == 0))
panel_hogar["POSESION_AMPLIA"] = panel_hogar["VIV_PROPIA"]

panel_hogar["gp_discapacidad"] = panel_hogar[["gp_discapacidad", "NEC_ACCES"]].max(axis=1)
GRUPOS = ["gp_mujer", "gp_joven", "gp_adulto_mayor", "gp_indigena", "gp_discapacidad"]
panel_hogar["grupo_prioritario"] = panel_hogar[GRUPOS].max(axis=1)

print(f"panel_hogar:    {panel_hogar.shape[0]:,} filas y {panel_hogar.shape[1]} columnas")
print(f"panel_vivienda: {panel_vivienda.shape[0]:,} filas y {panel_vivienda.shape[1]} columnas")
panel_hogar.head()

panel_hogar:    56,578 filas y 87 columnas
panel_vivienda: 55,147 filas y 438 columnas


,FOLIO,VIV_SEL,HOGAR,N_INF,P3A1_1,P3A1_2,P4_1,P4_2_2,P7_1,ENT,TLOC,EST_DIS,UPM_DIS,FACTOR,ENT_NOMBRE,TLOC_NOMBRE,AMBITO,NEC_DECLARADA,HOG_SEC,PROP_OTRA,tam_hogar,n_mayores60,n_perceptores,n_ing_no_reportado,ING_HOG,n_derechohab,hogar_derechohab,DERECHO_CRED,ING_IND,hogar_indigena,hogar_limitacion,ingreso_incompleto,ING_HOG_UMA,ING_5UMA,ING_IND_DH,ING_CONFIABLE,JEFE_SEXO,JEFE_EDAD,JEFE_NIV,JEFE_P2_8,JEFE_P3_1,JEFE_P3_3,JEFE_P3_5,MAYOR18,gp_mujer,gp_joven,gp_adulto_mayor,gp_indigena,gp_discapacidad,ING_HOG_PMM,ING_5UMA_PMM,n_demandas,DEM_RENTA,DEM_COMPRA,DEM_CONSTRUIR,DEM_CONSTR_TERR,PLAN_CRED,PLAN_AHORRO,P5_1,n_hogares,personas_por_cuarto,MAT_DET,MAT_REG,SIN_EXC,HACIN,REZAGO,CAR_SERV,SIN_COCINA,DANO_EST,NEC_MEJ_DECL,NEC_AMP_DECL,NEC_ACCES,tenencia,VIV_PROPIA,POSESION_DOC,TEN_IRREG,MULTIFAM,DANO_COMUN,UH_SOCIAL,CRED_VIGENTE,APOYO_PREVIO,SIN_VIV_PROPIA,DEM_VIV,NEC_VIV,NEC_VIV_OFICIAL,POSESION_AMPLIA,grupo_prioritario
0,0000001,01,1,1,2,<NA>,<NA>,2,2,1,1,2,1,206,Aguascalientes,"100,000 y más hab.",Urbano,0,0,0,3,3,1,1,"3,500.00",0,0,0,<NA>,1,0,1,1.33,1,0,0,1,61,3,6,2,4,4,1,0,0,1,1,0,"7,500.00",1,0,0,0,0,0,0,0,5,1,0.75,0,0,0,0,0,0,0,1,0,0,0,propia,1,1,0,0,0,0,0,0,0,0,0,0,1,1
1,0000001,02,1,2,2,<NA>,<NA>,2,2,1,1,2,1,206,Aguascalientes,"100,000 y más hab.",Urbano,0,0,0,3,2,2,0,"19,500.00",1,1,1,"6,500.00",0,0,0,7.38,0,1,1,1,74,2,5,1,4,4,1,0,0,1,0,0,"19,500.00",0,0,0,0,0,0,0,0,5,1,0.60,0,0,0,0,0,0,0,1,1,0,0,propia,1,1,0,0,0,0,0,0,0,0,0,0,1,1
2,0000001,03,1,1,2,<NA>,<NA>,1,1,1,1,2,1,206,Aguascalientes,"100,000 y más hab.",Urbano,0,0,1,1,0,0,1,0.00,0,0,0,<NA>,1,0,1,0.00,1,0,0,2,57,8,3,1,4,4,1,1,0,0,1,0,"3,500.00",1,0,0,0,0,0,0,0,1,1,0.20,0,0,0,0,0,0,0,1,0,0,0,rentada,0,0,0,0,0,0,0,0,1,0,1,0,0,1
3,0000001,04,1,3,2,<NA>,<NA>,3,2,1,1,2,1,206,Aguascalientes,"100,000 y más hab.",Urbano,0,0,0,8,1,0,0,0.00,0,0,0,<NA>,0,0,0,0.00,1,0,0,2,55,4,5,6,<NA>,<NA>,1,1,0,1,0,0,0.00,1,0,0,0,0,0,0,0,1,1,1.60,0,0,0,0,0,0,0,0,0,0,0,rentada,0,0,0,0,0,0,0,0,1,0,1,0,0,1
4,0000001,05,1,1,1,1,<NA>,1,2,1,1,2,1,206,Aguascalientes,"100,000 y más hab.",Urbano,1,0,0,3,1,2,0,"23,666.67",1,1,1,"15,000.00",0,0,0,8.96,0,0,1,2,86,8,4,4,<NA>,<NA>,1,1,0,1,0,0,"23,666.67",0,1,0,0,1,0,0,0,5,1,0.38,0,0,0,0,0,0,0,0,1,0,0,propia,1,1,0,0,0,0,0,0,0,1,1,0,1,1


#### Reglas de atención

Con las etiquetas se marca qué hogares pueden ser atendidos por cada tipo de apoyo del Anexo. Todos los tipos piden rezago habitacional (`REZAGO_HAB`, que es `REZAGO` o `NEC_VIV_OFICIAL`), salvo C7, que mira las áreas comunes del edificio.

- `POT_C1`, vivienda nueva en conjunto: necesidad de vivienda, jefe o jefa mayor de edad y nadie del hogar es dueño de otra vivienda.
- `POT_C2`, compra con crédito: quiere comprar, tiene derechohabiencia o piensa usar crédito, y nadie es dueño de otra vivienda.
- `POT_C3`, vivienda nueva en lote propio: materiales deteriorados, quiere construir y tiene terreno, o es hogar secundario en vivienda propia; además posee la vivienda o tiene terreno.
- `POT_C4A`, mejoramiento: materiales regulares, sin excusado, carencia de servicios, daño estructural o necesidad de reparar, posee la vivienda y no tiene materiales deteriorados (esos van a C3).
- `POT_C4B`, ampliación: hacinamiento, sin cocina o necesidad de ampliar, y posee la vivienda.
- `POT_C4C`, accesibilidad: necesita adaptaciones y además califica para C3, C4A o C4B. No se da sola.
- `POT_C7`, unidades habitacionales: multifamiliar con daños en áreas comunes y de interés social.
- `POT_C8`, regularización: rezago y tenencia irregular.

La posesión se toma de `ESCENARIO_POSESION` (amplio por defecto, con o sin escritura; estricto si se cambia a `POSESION_DOC`). Cada `OBJ_` es el mismo tipo pero con ingreso del hogar de 5 UMA o menos. `POT_TOTAL` y `OBJ_TOTAL` cuentan al hogar una sola vez aunque califique en varios tipos, y `N_TIPOS` dice en cuántos califica.

`APOYO_PREVIO`, `CRED_VIGENTE`, `PLAN_AHORRO`, `ING_IND_DH` e `ING_CONFIABLE` no entran a estas reglas. Se dejan en el panel para los escenarios de sensibilidad.

In [85]:
ph = panel_hogar

ph["REZAGO_HAB"] = ph["REZAGO"] | ph["NEC_VIV_OFICIAL"]
ph["POSESION"] = ph[ESCENARIO_POSESION]
rezago = ph["REZAGO_HAB"]
no_prop_otra = 1 - ph["PROP_OTRA"]

ph["POT_C1"] = rezago & ph["NEC_VIV"] & ph["MAYOR18"] & no_prop_otra
ph["POT_C2"] = rezago & ph["DEM_COMPRA"] & (ph["DERECHO_CRED"] | ph["PLAN_CRED"]) & no_prop_otra
ph["POT_C3"] = (rezago & (ph["MAT_DET"] | ph["DEM_CONSTR_TERR"] | (ph["HOG_SEC"] & ph["VIV_PROPIA"]))
                & (ph["POSESION"] | ph["DEM_CONSTR_TERR"]))
ph["POT_C4A"] = (rezago & (ph["MAT_REG"] | ph["SIN_EXC"] | ph["CAR_SERV"] | ph["DANO_EST"] | ph["NEC_MEJ_DECL"])
                 & ph["POSESION"] & (1 - ph["MAT_DET"]))
ph["POT_C4B"] = rezago & (ph["HACIN"] | ph["SIN_COCINA"] | ph["NEC_AMP_DECL"]) & ph["POSESION"]
ph["POT_C4C"] = ph["NEC_ACCES"] & (ph["POT_C3"] | ph["POT_C4A"] | ph["POT_C4B"])
ph["POT_C7"] = ph["MULTIFAM"] & ph["DANO_COMUN"] & ph["UH_SOCIAL"]
ph["POT_C8"] = rezago & ph["TEN_IRREG"]

TIPOS = ["C1", "C2", "C3", "C4A", "C4B", "C4C", "C7", "C8"]
for k in TIPOS:
    ph[f"OBJ_{k}"] = ph[f"POT_{k}"] & ph["ING_5UMA"]

POT = [f"POT_{k}" for k in TIPOS]
OBJ = [f"OBJ_{k}" for k in TIPOS]
ph["POT_TOTAL"] = ph[POT].max(axis=1)
ph["OBJ_TOTAL"] = ph[OBJ].max(axis=1)
ph["N_TIPOS"] = ph[POT].sum(axis=1)
ph["OBJ_TOTAL_PMM"] = ph["POT_TOTAL"] & ph["ING_5UMA_PMM"]

# Comprobación: hogares expandidos por tipo de atención
(ph[POT + ["POT_TOTAL"] + OBJ + ["OBJ_TOTAL"]].mul(ph["FACTOR"], axis=0).sum()).to_frame("hogares")

,hogares
POT_C1,4284616
POT_C2,299477
POT_C3,1934072
POT_C4A,6060590
POT_C4B,5476141
POT_C4C,1512013
POT_C7,172167
POT_C8,3214196
POT_TOTAL,9473755
OBJ_C1,3699614


#### Tabla de etiquetas

Una fila por columna de cada panel con su descripción. Las columnas de INEGI toman la pregunta del diccionario y sus categorías. Las que se crearon aquí toman la descripción de abajo. En las numéricas solo se da el rango observado.

In [86]:
DESCRIPCION = {
    "ENT_NOMBRE": "Nombre de la entidad federativa",
    "TLOC_NOMBRE": "Tamaño de localidad",
    "AMBITO": "Urbano (2,500 habitantes o más) o rural",
    "NEC_DECLARADA": "El hogar necesita o planea rentar, comprar o construir (P3A1_1 = 1)",
    "HOG_SEC": "Hogar secundario en la vivienda (P4_1 = 2)",
    "PROP_OTRA": "Alguien del hogar es dueño de otra vivienda (P7_1 = 1)",
    "tam_hogar": "Número de integrantes del hogar",
    "n_mayores60": "Integrantes de 60 años o más",
    "n_perceptores": "Integrantes con ingreso por trabajo mayor a cero",
    "n_ing_no_reportado": "Integrantes que no reportaron monto de ingreso",
    "ING_HOG": "Ingreso mensual por trabajo del hogar, solo montos reportados",
    "n_derechohab": "Integrantes con derecho a crédito de vivienda (P3_5 = 1, 2 o 3)",
    "hogar_derechohab": "Algún integrante con derecho a crédito de vivienda (P3_5 = 1, 2 o 3)",
    "DERECHO_CRED": "Algún integrante con crédito INFONAVIT o FOVISSSTE (P3_5 = 1 o 2)",
    "hogar_indigena": "Algún integrante habla lengua indígena o se considera indígena",
    "hogar_limitacion": "Algún integrante con limitación física o mental permanente (P3_1 = 7)",
    "ingreso_incompleto": "Al menos un integrante no reportó su ingreso",
    "ING_HOG_UMA": "Ingreso del hogar en UMA mensuales de 2020",
    "ING_5UMA": "Ingreso del hogar de 5 UMA o menos ($13,205.75)",
    "ING_HOG_PMM": "Ingreso mensual por trabajo del hogar con los faltantes imputados por PMM",
    "ING_5UMA_PMM": "Ingreso del hogar con PMM de 5 UMA o menos",
    "ING_IND": "Ingreso mensual por trabajo del integrante con crédito INFONAVIT o FOVISSSTE (el mayor)",
    "ING_IND_DH": "Hogar derechohabiente con ING_IND de 2 salarios mínimos o menos ($7,491.78)",
    "ING_CONFIABLE": "Hogar con al menos un perceptor y sin ingresos faltantes",
    "MAYOR18": "Jefe o jefa de 18 años o más",
    "gp_mujer": "Grupo prioritario: jefatura femenina",
    "gp_joven": "Grupo prioritario: jefe o jefa de 18 a 29 años",
    "gp_adulto_mayor": "Grupo prioritario: algún integrante de 60 años o más",
    "gp_indigena": "Grupo prioritario: algún integrante indígena",
    "gp_discapacidad": "Grupo prioritario: limitación permanente o vivienda que necesita adaptación",
    "n_demandas": "Viviendas que busca el hogar (registros en TDEMANDA)",
    "DEM_RENTA": "Quiere rentar (P3A4 = 1)",
    "DEM_COMPRA": "Quiere comprar vivienda nueva o usada (P3A4 = 2 o 3)",
    "DEM_CONSTRUIR": "Quiere construir (P3A4 = 4)",
    "DEM_CONSTR_TERR": "Quiere construir y ya tiene terreno (P3A4 = 4 y P3A7 = 1)",
    "PLAN_CRED": "Piensa usar crédito INFONAVIT, FOVISSSTE, privado u otra institución pública",
    "PLAN_AHORRO": "Piensa usar recursos propios para la vivienda (P3A8 = 1)",
    "n_hogares": "Hogares que viven en la vivienda",
    "personas_por_cuarto": "Residentes entre total de cuartos con cocina (P1_1 / P4_10A)",
    "MAT_DET": "Paredes o techo de material deteriorado",
    "MAT_REG": "Paredes o techo de material regular, o piso de tierra",
    "SIN_EXC": "Sin excusado (P4_11 = 2)",
    "HACIN": "Más de 2.5 personas por cuarto",
    "REZAGO": "Rezago habitacional: MAT_DET, MAT_REG, SIN_EXC o HACIN",
    "CAR_SERV": "Carencia de agua, drenaje, luz o cocina con leña sin chimenea",
    "SIN_COCINA": "Sin cuarto para cocinar (P4_9 = 2)",
    "DANO_EST": "Problemas estructurales (P4_25_1 a P4_25_5)",
    "NEC_MEJ_DECL": "Dice necesitar reparaciones o mantenimiento",
    "NEC_AMP_DECL": "Dice necesitar construir cuarto, baño u otros espacios",
    "NEC_ACCES": "Necesita adaptaciones para personas con discapacidad (P6_7)",
    "tenencia": "Tenencia de la vivienda (P5_1) en texto",
    "VIV_PROPIA": "Vivienda propia o propia que se está pagando (P5_1 = 4 o 5)",
    "POSESION_DOC": "Propia con escritura a nombre del dueño (P5_7 = 1)",
    "TEN_IRREG": "Intestada o en litigio, o propia sin escritura a su nombre",
    "MULTIFAM": "Departamento, vecindad o cuarto de azotea (P4_3 = 4, 5 o 6)",
    "DANO_COMUN": "Daños en pasillos, escaleras o elevador del edificio",
    "UH_SOCIAL": "Multifamiliar comprada con crédito INFONAVIT o FOVISSSTE (P5_15_01 o P5_15_02 = 1)",
    "CRED_VIGENTE": "Deuda vigente con el crédito INFONAVIT de la vivienda (P5_18_1 = 1)",
    "APOYO_PREVIO": "Material gratis del gobierno en 2019 o 2020 para la vivienda (P5_33, P5_34, P5_46, P5_47)",
    "SIN_VIV_PROPIA": "Vivienda rentada, prestada u otra, o hogar secundario",
    "DEM_VIV": "Necesidad declarada con intención de rentar, comprar o construir",
    "NEC_VIV": "Necesidad de vivienda: SIN_VIV_PROPIA o DEM_VIV",
    "NEC_VIV_OFICIAL": "Hogar secundario en vivienda sin rezago (referencia CONAVI)",
    "POSESION_AMPLIA": "Posee la vivienda, con o sin escritura",
    "grupo_prioritario": "Pertenece a algún grupo prioritario",
    "REZAGO_HAB": "Rezago habitacional o hogar secundario sin rezago (REZAGO o NEC_VIV_OFICIAL)",
    "POSESION": "Posesión usada en las reglas, según ESCENARIO_POSESION",
    "POT_C1": "Potencial C1, vivienda nueva en conjunto",
    "POT_C2": "Potencial C2, compra de vivienda con crédito",
    "POT_C3": "Potencial C3, vivienda nueva en lote propio",
    "POT_C4A": "Potencial C4a, mejoramiento",
    "POT_C4B": "Potencial C4b, ampliación",
    "POT_C4C": "Potencial C4c, accesibilidad (complementa C3, C4a o C4b)",
    "POT_C7": "Potencial C7, unidades habitacionales",
    "POT_C8": "Potencial C8, regularización de la tenencia",
    "OBJ_C1": "Objetivo C1: POT_C1 con ingreso de 5 UMA o menos",
    "OBJ_C2": "Objetivo C2: POT_C2 con ingreso de 5 UMA o menos",
    "OBJ_C3": "Objetivo C3: POT_C3 con ingreso de 5 UMA o menos",
    "OBJ_C4A": "Objetivo C4a: POT_C4A con ingreso de 5 UMA o menos",
    "OBJ_C4B": "Objetivo C4b: POT_C4B con ingreso de 5 UMA o menos",
    "OBJ_C4C": "Objetivo C4c: POT_C4C con ingreso de 5 UMA o menos",
    "OBJ_C7": "Objetivo C7: POT_C7 con ingreso de 5 UMA o menos",
    "OBJ_C8": "Objetivo C8: POT_C8 con ingreso de 5 UMA o menos",
    "POT_TOTAL": "Población potencial: califica en al menos un tipo de atención",
    "OBJ_TOTAL": "Población objetivo: es objetivo en al menos un tipo de atención",
    "OBJ_TOTAL_PMM": "Población objetivo con el ingreso imputado por PMM",
    "N_TIPOS": "Número de tipos de atención en los que califica (traslape)",
}

dicc_unico = dicc[dicc["tabla"] == dicc.groupby("columna")["tabla"].transform("first")]
pregunta = dicc_unico.drop_duplicates("columna").set_index("columna")["pregunta"]
es_numerica = dicc_vars.drop_duplicates("columna").set_index("columna")["es_numerica"]
categorias = (dicc_unico["codigo"] + "=" + dicc_unico["etiqueta"].fillna("")).groupby(dicc_unico["columna"]).agg("; ".join)


def rango(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return f"de {s.min():,.2f} a {s.max():,.2f}" if len(s) else "sin datos"


def tabla_etiquetas(panel):
    filas = []
    for col in panel.columns:
        original = col.replace("JEFE_", "")
        s = panel[col]
        if original in pregunta.index:
            origen = "INEGI"
            descripcion = ("Jefe o jefa: " if col != original else "") + pregunta[original]
            valores = rango(s) if es_numerica[original] else categorias[original]
        else:
            origen = "creada"
            descripcion = DESCRIPCION[col]
            if s.dtype == object or pd.api.types.is_string_dtype(s):
                valores = "; ".join(sorted(s.dropna().astype(str).unique())[:12])
            elif set(s.dropna().unique()) <= {0, 1}:
                valores = "0=No; 1=Sí"
            else:
                valores = rango(s)
        filas.append([col, origen, descripcion, valores, round(100 * s.isna().mean(), 1)])
    return pd.DataFrame(filas, columns=["columna", "origen", "descripcion", "valores", "vacios_%"])


etiquetas_hogar = tabla_etiquetas(panel_hogar)
etiquetas_vivienda = tabla_etiquetas(panel_vivienda)
etiquetas_hogar

,columna,origen,descripcion,valores,vacios_%
0,FOLIO,INEGI,Folio,"de 1.00 a 8,301.00",0.00
1,VIV_SEL,INEGI,Número de Vivienda seleccionada,de 1.00 a 24.00,0.00
2,HOGAR,INEGI,Número de Hogar en la vivienda,de 1.00 a 5.00,0.00
3,N_INF,INEGI,Número del informante del hogar,de 1.00 a 13.00,0.00
4,P3A1_1,INEGI,"3a.1 ¿Alguna de las personas de este hogar tiene necesidad o está planeando rentar, co...",1=Sí; 2=No; 9=No sabe,0.00
5,P3A1_2,INEGI,"En este hogar, ¿cuántas viviendas necesitan o están planeando rentar, comprar o constr...",de 1.00 a 5.00,78.50
6,P4_1,INEGI,"4.1 ¿En su hogar se localiza la dueña o el dueño, la persona responsable del contrato ...",1=Sí (es hogar principal); 2=No (es hogar secundario); b=Blanco por secuencia,95.30
7,P4_2_2,INEGI,Número de renglón del informante de vivienda del hogar principal,de 1.00 a 14.00,2.50
8,P7_1,INEGI,"7.1 ¿Alguna persona de este hogar es propietaria de otras viviendas, independientement...",1=Sí; 2=No,0.00
9,ENT,INEGI,ENTIDAD FEDERATIVA,de 1.00 a 32.00,0.00


## 9. Guardar los paneles sin expandir

Cada archivo se guarda y se vuelve a leer para revisar que tenga las mismas filas y columnas, que la llave no se repita y que el `FACTOR` sume lo mismo.

In [87]:
def guardar(df, nombre, llave):
    ruta = OUT / nombre
    df.to_csv(ruta, index=False, encoding="utf-8-sig")

    # Comprobación: lo que se lee es igual a lo que se escribió
    leido = pd.read_csv(ruta, dtype={c: str for c in LLAVES}, encoding="utf-8-sig", low_memory=False)
    assert leido.shape == df.shape, f"{nombre}: cambió el tamaño"
    assert not leido.duplicated(llave).any(), f"{nombre}: llaves repetidas"
    if "FACTOR" in df.columns:
        assert leido["FACTOR"].sum() == df["FACTOR"].sum(), f"{nombre}: el FACTOR no suma igual"
    print(f"{nombre}: {len(df):,} filas y {df.shape[1]} columnas, guardado y revisado")


DISENO = ["ENT", "TLOC", "EST_DIS", "UPM_DIS", "FACTOR"]

creadas_hogar = etiquetas_hogar.loc[etiquetas_hogar["origen"] == "creada", "columna"].tolist()
creadas_vivienda = etiquetas_vivienda.loc[etiquetas_vivienda["origen"] == "creada", "columna"].tolist()

analisis_hogar = panel_hogar[LLAVE_HOG + DISENO + creadas_hogar]
analisis_vivienda = panel_vivienda[LLAVE_VIV + DISENO + creadas_vivienda]

etiquetas_hogar["en_version_analisis"] = bandera(etiquetas_hogar["columna"].isin(analisis_hogar.columns))
etiquetas_vivienda["en_version_analisis"] = bandera(etiquetas_vivienda["columna"].isin(analisis_vivienda.columns))

guardar(panel_hogar, "panel_hogar_completo.csv", LLAVE_HOG)
guardar(panel_vivienda, "panel_vivienda_completo.csv", LLAVE_VIV)
guardar(analisis_hogar, "panel_hogar_analisis.csv", LLAVE_HOG)
guardar(analisis_vivienda, "panel_vivienda_analisis.csv", LLAVE_VIV)

with pd.ExcelWriter(OUT / "etiquetas_paneles.xlsx") as xls:
    etiquetas_hogar.to_excel(xls, sheet_name="panel_hogar", index=False)
    etiquetas_vivienda.to_excel(xls, sheet_name="panel_vivienda", index=False)
print("etiquetas_paneles.xlsx guardado")

panel_hogar_completo.csv: 56,578 filas y 109 columnas, guardado y revisado
panel_vivienda_completo.csv: 55,147 filas y 438 columnas, guardado y revisado
panel_hogar_analisis.csv: 56,578 filas y 95 columnas, guardado y revisado
panel_vivienda_analisis.csv: 55,147 filas y 29 columnas, guardado y revisado
etiquetas_paneles.xlsx guardado
